# SearchLite-RL headroom study: does GRPO help an SFT agent where SFT has room to improve?
Branch `experiment/headroom-grpo`. Run cells in order. Every code cell is `%%bash` with `set -euo pipefail`; a gate that fails exits with code 3 and the measured reason (do not continue past it, paste the output). Nothing is deleted or overwritten; outputs go to Google Drive (`MyDrive/SearchLite-RL/study_hard`, plus the existing `study_sft`). Re-running resumes.
Estimated free-Colab (L4) cost: cell 4 ~5 min (no GPU needed), cell 5 ~2.5 h (stages are gated and resumable), cell 6 ~10 min.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Pilot-project ]; then cd Pilot-project && git fetch -q && git checkout -q experiment/headroom-grpo && git pull -q; else git clone -q -b experiment/headroom-grpo https://github.com/aasritha-oql7872/Pilot-project.git; fi
cd /content/Pilot-project
python -c "import trl,peft,transformers" 2>/dev/null && echo "dependencies already installed" || { pip install -q -r requirements.txt; pip uninstall -y -q torchvision torchaudio torchao; echo "INSTALLED: restart the session (Runtime > Restart session), then continue with the next cell"; }
git log --oneline -3

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
D=/content/drive/MyDrive/SearchLite-RL
mkdir -p outputs $D/study_sft $D/study_hard
for s in study_sft study_hard; do ln -sfn $D/$s outputs/$s; done
mkdir -p outputs/main_v2
[ -f outputs/main_v2/adapter/adapter_config.json ] || cp -rn $D/outputs/main_v2/adapter outputs/main_v2/ 2>/dev/null || cp -rn $D/main_v2/adapter outputs/main_v2/ 2>/dev/null || echo "condition A adapter not on Drive (M1 would be retrained by run_sft_study.sh)"
nvidia-smi -L
ls outputs/study_sft | head -40
python -m pytest -q 2>&1 | tail -3

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# PHASE 1/2 AUDIT of the existing artifacts (no GPU): were C and D trained independently, and why are they identical?
S=outputs/study_sft; O=outputs/study_hard
python -m src.verify_reported $S || true      # reported vs saved dev numbers (MISMATCH lines are findings, not errors)
python -m src.audit_lineage B=$S/B C=$S/C D=$S/D --out $O/lineage_BCD.json
python -m src.train_diag $S/C --init $S/B/adapter --out $O/diag_C_existing.json
python -m src.train_diag $S/D --init $S/B/adapter --out $O/diag_D_existing.json
python -m src.traj_stats $S/C $S/D --out $O/traj_stats_CD_existing.json
echo "--- training logs: which steps had a gradient? ---"
python - <<'PY'
import csv
for r in "CD":
    rows=list(csv.DictReader(open(f"outputs/study_sft/{r}/train_metrics.csv")))
    nz=[(int(x["step"]),round(float(x["grad_norm"]),4),x.get("searchlite/twohop_searches"),x["frac_reward_zero_std"]) for x in rows if float(x["grad_norm"])>0]
    print(r,"steps with nonzero grad (step, grad_norm, twohop_searches, frac_zero_std):",nz)
PY

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# PHASE 3-5: headroom check -> signal check -> matched GRPO runs -> hard_dev tables. Gated (exit 3 = stop and report).
bash run_headroom_study.sh 2>&1 | tee -a outputs/study_hard/run_$(date +%Y%m%d_%H%M).log | tail -80
cat outputs/study_hard/summary_hard_dev.md

In [ ]:
%%bash
set -euo pipefail
cd /content/Pilot-project
# Look at the evidence before choosing the headline model (hard_dev only), then FREEZE it in the next cell.
O=outputs/study_hard
cat $O/decision_headroom.json $O/decision_signal.json
cat $O/summary_dev_regression.md $O/summary_twohop_dev_regression.md
python - <<'PY'
import json
d=json.load(open("outputs/study_hard/traj_stats.json"))
for k,v in d.items(): print(k, "informative groups:",v["all_groups_nonzero_std"],"twohop second-search:",v["twohop"]["second_search"],"bonus-created variance:",v["twohop"]["bonus_created_variance"])
PY

In [ ]:
# FREEZE (edit the text, then run ONCE). Pre-registered rule: a GRPO variant replaces M2 only if its mean hard_dev accuracy
# beats M2 by >= 5 points AND dev / twohop_dev each drop by <= 3 points; otherwise the headline is M2.
# !echo "headline: M2 (SFT only); applied the pre-registered rule to hard_dev, $(date -u +%FT%TZ)" > /content/Pilot-project/outputs/study_hard/FROZEN_HARD

In [ ]:
# LOCKED hard_final evaluation, once.
# %%bash
# set -euo pipefail
# cd /content/Pilot-project
# bash run_hard_final.sh 2>&1 | tee -a outputs/study_hard/final_$(date +%Y%m%d_%H%M).log | tail -40
# cat outputs/study_hard/summary_hard_final.md